## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
import os

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("../twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("../twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"  # Placeholder; no OpenAI API key needed
)

response = client.chat.completions.create(model="qwen2.5-coder:1.5b", messages=messages)


print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = client.chat.completions.create(model="qwen2.5-coder:1.5b", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = client.chat.completions.create(model="qwen2.5-coder:1.5b", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = client.chat.completions.create(model="qwen2.5-coder:1.5b", messages=messages)
print(response.choices[0].message.content)

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
If the visitor gives an email address, you MUST call record_email_tool.
Do not only acknowledge it in text.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
    {"role": "assistant", "content": "If the visitor gives an email address, you MUST call record_email_tool.Do not only acknowledge it in text."}
]

In [ ]:
response = client.chat.completions.create(model="qwen2.5-coder:1.5b", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="qwen2.5-coder:1.5b", messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=False)

# And now - TOOLS!

Let's start with a function...

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("yozna@testy.com")

## Step 1 - write some json to describe the tool


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="llama3.2:latest", messages=messages, tools=tools)
         
    if response.choices[0].message.tool_calls:
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)

               
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = client.chat.completions.create(model="llama3.2:latest", messages=messages, tools=tools)
    
    msg = response.choices[0]
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=False)

## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="llama3.2:latest", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = client.chat.completions.create(model="llama3.2:latest", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=False)

Multiple LLM Calls 

In [ ]:
def evaluate_result(user_message, agent_response, tool_called, tool_result):

    evaluator_prompt = f"""
    You are an evaluator responsible for assessing an AI email assistant.

    Evaluate the assistant based on these criteria:

    1. Did the assistant correctly understand the user's request?
    2. If the user provided an email address, was the email recording tool called?
    3. Was the correct email address passed to the tool?
    4. Was the tool execution successful?
    5. Did the assistant respond appropriately?

    User Message:
    {user_message}

    Agent Response:
    {agent_response}

    Tool Called:
    {tool_called}

    Tool Execution Result:
    {tool_result}

    Return:
    PASS or FAIL
    Followed by a short explanation.
    """

    evaluation = client.chat.completions.create(
        model="llama3.2:latest",
        messages=[
            {
                "role": "system",
                "content": "You are a strict AI response evaluator."
            },
            {
                "role": "user",
                "content": evaluator_prompt
            }
        ]
    )

    return evaluation.choices[0].message.content

def chat(message, history):

    messages = [
        {"role": "system", "content": system_prompt}
    ] + history + [
        {"role": "user", "content": message}
    ]

    response = client.chat.completions.create(
        model="llama3.2:latest",
        messages=messages,
        tools=tools
    )

    tool_called = False
    tool_result = "No tool was called"

    if response.choices[0].message.tool_calls:

        assistant_message = response.choices[0].message
        tool_call = assistant_message.tool_calls[0]

        email = json.loads(
            tool_call.function.arguments
        ).get("email")

        record_email_tool(email)

        tool_called = True
        tool_result = f"Email {email} recorded successfully"

        messages.append(assistant_message)

        messages.append({
            "role": "tool",
            "content": "Email recorded",
            "tool_call_id": tool_call.id
        })

        response = client.chat.completions.create(
            model="llama3.2:latest",
            messages=messages,
            tools=tools
        )

    agent_response = response.choices[0].message.content

    # Second LLM call: Evaluate the agent's performance
    evaluation = evaluate_result(
        message,
        agent_response,
        tool_called,
        tool_result
    )

    print("\n----- EVALUATION -----")
    print(evaluation)

    return agent_response

gr.ChatInterface(chat).launch(inbrowser=False)